# B1.4 · One skeleton, four oracles — and whether any of it is true

**Function B — Application Security with an AI SDLC → The Agentic Harness**  ·  *Both directions*

Builds on **[B1.3 · Which model runs it, and who checks the checker](https://spbreed.github.io/cyber-commons/lessons/B1.3.html)**.

| | |
|---|---|
| Tools used | Semgrep |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

A harness that is right 80% of the time has a pass@5 of 99.97% and a pass^5 of 33%. Both are true. Quoting the first for something that runs unattended is where most harness claims quietly go wrong, and the number nobody quotes at all is cost per confirmed finding.

> **At CyberTravels.** Alex has to tell someone whether the review pipeline can be left running unattended over a weekend. That is a pass^k question, and the number he has is pass@k.

## 2 · The framework

```
   one skeleton                       supplied by the domain
   +---------------------------+      +----------------------+
   | plan -> act -> verify     | <--- | the ORACLE           |
   |            -> stop        |      | the BLAST RADIUS     |
   +---------------------------+      +----------------------+

     SAST     reachability + failing test      read source: free
     threat   diff against the last model      read config: free
     DAST     observed change in a response    hit a replica: small
     pentest  a shell, a row, a file           hit prod: an incident

   then grade it:
     conformance  ~100% by construction   <- build health
     accuracy     the number that means something

     pass@k  succeeded at least once   <- someone is checking
     pass^k  succeeded k out of k      <- nobody is
```

### One skeleton, four oracles

Teams build a SAST harness, then a threat-modelling harness, then a DAST
harness, then a pentest harness — and re-decide loop control, budgets, retries
and verification four times. The loops end up nearly identical and the four
teams each get one thing wrong in their own way.

They are the same skeleton: plan a candidate, act on it, verify it, stop. Two
things differ, and both belong to the domain rather than to the loop:

**The oracle** — what decides a candidate is real. Reachability plus a failing
test for static analysis. A diff against the previous model for threat
modelling. An observed change in a response for DAST. A shell, a row or a file
for a pentest. The oracle is the whole value of the harness; everything else is
plumbing you have already built.

**The blast radius** — what acting costs if the candidate is wrong. Reading
CyberTravels' source costs nothing. A request to a replica costs a little.
Running an exploit against the live booking API costs an incident, so it needs
an authorisation the loop cannot grant itself.

### Then the question the chapter has been walking towards

Four stages decide whether the output is worth acting on: **ingest** (did it
parse), **path matching** (is the finding about the file we asked about),
**expert proxy** (is it right, scored 0 / 0.5 / 1), and **dual judges** (is the
reasoning sound, aggregated by MIN).

> **Conformance** is schema validity. With structured output it is ~100% by
> construction. It is a build-health signal.
>
> **Accuracy** is correctness. It is the number that means something.

Quoting conformance as quality — "our harness scores 100%" — is the single most
common way a security evaluation misleads its own sponsors. And one
implementation detail silently randomises everybody's results: **file matching
must use parent directory plus filename, never the bare basename**, because
public corpora reuse `1.py` and `3.c` across every CWE directory.

### And one run tells you nothing

**pass@k** — succeeded at least once in k attempts. Right when you can cheaply
check which attempt was correct and keep it.

**pass^k** — succeeded every time, k out of k. Right when the run is autonomous
and nobody is checking, which describes every harness that files a ticket, gates
a merge or closes an alert.

A harness at 80% per-run reliability has a pass@5 of 99.97% and a pass^5 of
33%. Both numbers are true. Quoting the first for a system that runs unattended
is where most harness claims quietly go wrong — and the cost that matters is not
per run but **per confirmed finding**, which is the only figure that survives
contact with a finance conversation.

## 3 · One skeleton, and the two things a domain supplies

In [ ]:
DOMAINS = {
 "sast":         {"reads": "source at a commit",     "blast": "read-only"},
 "threat model": {"reads": "architecture and IaC",   "blast": "read-only"},
 "dast":         {"reads": "a running replica",      "blast": "replica-write"},
 "pentest":      {"reads": "an owned host in scope", "blast": "live-action"},
}
ORACLES = {
 "sast":         ("reachable from an entrypoint AND a failing test",
                  lambda e: e["reachable"] and e["failing_test"]),
 "threat model": ("present in the new model and absent from the old",
                  lambda e: e["in_new"] and not e["in_old"]),
 "dast":         ("response differs from the control request",
                  lambda e: e["response_differs"]),
 "pentest":      ("an artefact that should not have been obtainable",
                  lambda e: e["artefact"] is not None),
}
AUTHORISED = {"read-only", "replica-write"}      # live-action needs a signed scope

def harness(domain, candidates, oracle, budget=6, scope_signed=False):
    """The skeleton. Identical for all four domains."""
    blast = DOMAINS[domain]["blast"]
    if blast not in AUTHORISED and not scope_signed:
        return {"domain": domain, "refused": "live action without a signed scope",
                "confirmed": [], "steps": 0}
    confirmed, steps = [], 0
    for c in sorted(candidates, key=lambda c: c["id"]):
        if steps >= budget:
            break
        steps += 1
        if oracle(c["evidence"]):
            confirmed.append(c)
    return {"domain": domain, "refused": None, "confirmed": confirmed, "steps": steps}

for d in sorted(DOMAINS):
    print(f"{d:14s}{DOMAINS[d]['blast']:14s}oracle: {ORACLES[d][0]}")

## 4 · Four domains through the same loop

In [ ]:
CANDIDATES = {
 "sast": [
  {"id": "unit_07 CWE-89 in build_query", "real": True,
   "evidence": {"reachable": True,  "failing_test": True}},
  {"id": "unit_12 CWE-89 in log_line",    "real": False,
   "evidence": {"reachable": False, "failing_test": False}},
  {"id": "unit_31 CWE-22 in export_path", "real": True,
   "evidence": {"reachable": True,  "failing_test": True}}],
 "threat model": [
  {"id": "worker -> db crosses trust 0 to 2", "real": True,
   "evidence": {"in_new": True,  "in_old": False}},
  {"id": "component web renamed to frontend", "real": False,
   "evidence": {"in_new": True,  "in_old": True}},
  {"id": "/admin/export on an untrusted entry", "real": True,
   "evidence": {"in_new": True,  "in_old": False}}],
 "dast": [
  {"id": "GET /v1/users returns 200 unauthenticated", "real": True,
   "evidence": {"response_differs": True}},
  {"id": "banner discloses framework version",        "real": False,
   "evidence": {"response_differs": False}},
  {"id": "POST /report reflects payload unencoded",   "real": True,
   "evidence": {"response_differs": True}}],
 "pentest": [
  {"id": "password auth permits spraying, no lockout", "real": True,
   "evidence": {"artefact": "session as svc-reports"}},
  {"id": "expired certificate on the partner CDN",     "real": False,
   "evidence": {"artefact": None}},
  {"id": "/backup listing exposes a database dump",    "real": True,
   "evidence": {"artefact": "orders.sql, 41 MB"}}],
}

def precision(result, candidates):
    if not result["confirmed"]:
        return None
    return sum(c["real"] for c in result["confirmed"]) / len(result["confirmed"])

print(f"{'domain':14s}{'confirmed':>10}{'precision':>11}  note")
for d in sorted(DOMAINS):
    r = harness(d, CANDIDATES[d], ORACLES[d][1], scope_signed=True)
    p = precision(r, CANDIDATES[d])
    print(f"{d:14s}{len(r['confirmed']):>10}{p:>11.2f}  {ORACLES[d][0][:38]}")

r = harness("pentest", CANDIDATES["pentest"], ORACLES["pentest"][1])
print(f"\npentest without a signed scope: {r['refused']}")
assert r["confirmed"] == []

## 5 · Where it breaks — the oracle everyone reaches for

Every one of those four oracles is a fact about the target. The tempting fifth is the model's own opinion, because it is the only one that works in every domain without being built.

In [ ]:
def model_oracle(evidence):
    """The model read the evidence and is confident. Confidence is not an oracle."""
    return True

print(f"{'domain':14s}{'confirmed':>10}{'precision':>11}")
for d in sorted(DOMAINS):
    r = harness(d, CANDIDATES[d], model_oracle, scope_signed=True)
    print(f"{d:14s}{len(r['confirmed']):>10}{precision(r, CANDIDATES[d]):>11.2f}")

total = sum(len(CANDIDATES[d]) for d in CANDIDATES)
real = sum(c["real"] for d in CANDIDATES for c in CANDIDATES[d])
print(f"\nevery one of {total} candidates confirmed; {real} of them are real.")
print("Precision 0.67 in every domain, and it looks like 1.00 from inside the")
print("harness, because the thing producing the finding is also the thing")
print("agreeing with it.")
assert all(precision(harness(d, CANDIDATES[d], model_oracle, scope_signed=True),
                     CANDIDATES[d]) < 1.0 for d in DOMAINS)

## 6 · The control — classify the oracle, gate on the class

In [ ]:
ORACLE_CLASS = {
 "sast":         "deterministic",   # re-runs to the same answer on the same commit
 "threat model": "deterministic",
 "dast":         "observational",   # a real observation, but of a system that moves
 "pentest":      "observational",
 "model":        "judgement",       # not re-checkable, not falsifiable
}
MAY_FILE = {"deterministic", "observational"}

def dispatch(domain, oracle_name, result):
    cls = ORACLE_CLASS[oracle_name]
    return {"domain": domain, "oracle_class": cls,
            "action": "file the finding" if cls in MAY_FILE else "queue for a human",
            "count": len(result["confirmed"])}

for d in sorted(DOMAINS):
    good = harness(d, CANDIDATES[d], ORACLES[d][1], scope_signed=True)
    bad  = harness(d, CANDIDATES[d], model_oracle, scope_signed=True)
    print(dispatch(d, d, good))
    print(dispatch(d, "model", bad))

print()
print("The skeleton did not change once across four domains. What changed was")
print("the oracle and the blast radius - which is the whole argument for")
print("building the loop once and never again.")
assert dispatch("sast", "model", harness("sast", CANDIDATES["sast"], model_oracle,
                scope_signed=True))["action"] == "queue for a human"

## 7 · Grading it — conformance is not accuracy

Four oracles, one skeleton. Now the question the whole chapter has been walking towards: is any of what it produces true?

In [ ]:
import json
from dataclasses import dataclass, field

@dataclass
class Answer:
    qid: str; cwe: str = ""; file: str = ""; line: int = 0; rationale: str = ""
    REQUIRED = ("qid", "cwe", "file", "rationale")

    @classmethod
    def parse(cls, raw):
        try:
            d = json.loads(raw)
        except json.JSONDecodeError as e:
            return None, f"non-conforming: not JSON ({e.msg})"
        missing = [k for k in cls.REQUIRED if not d.get(k)]
        if missing:
            return None, f"non-conforming: missing {missing}"
        return cls(str(d["qid"]), str(d["cwe"]).upper(), str(d["file"]),
                   int(d.get("line", 0)), str(d["rationale"])), "conforming"

@dataclass
class Truth:
    qid: str; cwe: str; file: str; line: int = 0

TRUTHS = {
 "q1": Truth("q1", "CWE-89",  "CWE-89/1.py"),
 "q2": Truth("q2", "CWE-78",  "CWE-78/1.py"),
 "q3": Truth("q3", "CWE-22",  "CWE-22/3.c"),
 "q4": Truth("q4", "CWE-798", "CWE-798/2.py"),
}
ANSWERS = {
 "q1": '{"qid":"q1","cwe":"CWE-89","file":"CWE-89/1.py","line":2,'
       '"rationale":"user input is concatenated into the query string"}',
 "q2": '{"qid":"q2","cwe":"CWE-89","file":"CWE-78/1.py","line":3,'
       '"rationale":"untrusted input reaches a shell"}',
 "q3": '{"qid":"q3","cwe":"CWE-22","file":"CWE-89/1.py","line":1,'
       '"rationale":"path built from user input"}',
 "q4": 'I think this file contains a hardcoded credential.',
}
for qid, raw in ANSWERS.items():
    _, note = Answer.parse(raw)
    print(f"{qid}: {note}")

## 8 · Stage 2 — the one line that decides whether this is a benchmark

Public corpora reuse filenames across directories. Match on the basename and you score an answer about CWE-79 against the ground truth for CWE-89 — and your accuracy becomes a random variable.

In [ ]:
def path_key(path):
    """Parent directory + filename. NEVER the bare basename."""
    parts = [p for p in path.replace("\\", "/").split("/") if p not in ("", ".")]
    return "/".join(parts[-2:]) if len(parts) > 1 else (parts[-1] if parts else "")

def basename(path):
    return path.replace("\\", "/").split("/")[-1]

pairs = [("CWE-89/1.py", "CWE-79/1.py"), ("a/CWE-22/3.c", "b/CWE-78/3.c")]
print(f"{'pair':34s}{'basename match':17s}path_key match")
print("-" * 66)
for a, b in pairs:
    print(f"{a + '  vs  ' + b:34s}{str(basename(a)==basename(b)):17s}"
          f"{path_key(a)==path_key(b)}")
print("\nq3 answered about CWE-89/1.py when the truth is CWE-22/3.c.")
print(f"   basename says match: {basename('CWE-89/1.py') == basename('CWE-22/3.c')}")
print(f"   path_key says match: {path_key('CWE-89/1.py') == path_key('CWE-22/3.c')}")

## 9 · Stages 3 and 4 — expert proxy and two judges

Half credit is not politeness. "Right file, wrong vulnerability class" is a genuinely different failure from "wrong file entirely", and averaging them away hides which one your harness is making.

Two judges aggregated by **MIN**, not mean — judges exist to catch each other, and averaging lets the lenient one carry the strict one's failures.

In [ ]:
def path_match(a, t): return path_key(a.file) == path_key(t.file)
def cwe_match(a, t):  return a.cwe == t.cwe.upper()

def expert_proxy(a, t):
    if not path_match(a, t): return 0.0
    return 1.0 if cwe_match(a, t) else 0.5

MECHANISM = ("concatenat", "unsanitis", "unsanitiz", "untrusted", "user input",
             "interpolat", "taint", "unvalidated")
def judge_strict(a, t):
    if expert_proxy(a, t) < 1.0: return 0.0
    return 1.0 if any(w in a.rationale.lower() for w in MECHANISM) else 0.5
def judge_lenient(a, t):
    return 1.0 if cwe_match(a, t) else 0.0

@dataclass
class Report:
    total: int = 0; conforming: int = 0
    expert_sum: float = 0.0; judge_sum: float = 0.0
    failures: list = field(default_factory=list)
    @property
    def conformance(self): return self.conforming / self.total if self.total else 0
    @property
    def expert_accuracy(self): return self.expert_sum / self.total if self.total else 0
    @property
    def judge_accuracy(self): return self.judge_sum / self.total if self.total else 0
    def render(self):
        return (f"  questions            {self.total}\n"
                f"  conformance          {self.conformance:.4f}   "
                f"← schema validity. Structural. NOT quality.\n"
                f"  expert accuracy      {self.expert_accuracy:.4f}   ← correctness\n"
                f"  judge accuracy (MIN) {self.judge_accuracy:.4f}\n"
                f"  failures             {len(self.failures)}")

def evaluate(answers, truths):
    rep = Report(total=len(truths))
    for qid, t in truths.items():
        a, note = Answer.parse(answers.get(qid, ""))
        if a is None:
            rep.failures.append((qid, "1-ingest", note)); continue
        rep.conforming += 1
        e = expert_proxy(a, t)
        j = min(judge_strict(a, t), judge_lenient(a, t))
        rep.expert_sum += e; rep.judge_sum += j
        if e < 1.0:
            why = ("wrong file" if not path_match(a, t)
                   else f"right file, wrong class (said {a.cwe}, truth {t.cwe})")
            rep.failures.append((qid, "3-expert", why))
    return rep

rep = evaluate(ANSWERS, TRUTHS)
print(rep.render())
print("\nfailures:")
for qid, stage, why in rep.failures:
    print(f"   {qid}  [{stage}]  {why}")

## 10 · The control — never report one number

Here is what happens when the harness is upgraded to emit structured output. Conformance goes to 1.0. Nothing about its capability changed.

In [ ]:
STRUCTURED = dict(ANSWERS)
STRUCTURED["q4"] = ('{"qid":"q4","cwe":"CWE-798","file":"CWE-798/2.py","line":1,'
                    '"rationale":"a credential is hardcoded"}')
rep2 = evaluate(STRUCTURED, TRUTHS)
print("after adding structured output:")
print(rep2.render())
print(f"\nconformance  {rep.conformance:.2f} → {rep2.conformance:.2f}   "
      f"(+{rep2.conformance-rep.conformance:.2f})")
print(f"expert acc   {rep.expert_accuracy:.2f} → {rep2.expert_accuracy:.2f}   "
      f"(+{rep2.expert_accuracy-rep.expert_accuracy:.2f})")
print("\nA press release could truthfully say 'conformance improved to 100%'.")
print("The harness still gets half the questions wrong.")

def gameable(answers):
    parsed = [Answer.parse(r)[0] for r in answers.values()]
    ok = [p for p in parsed if p]
    cwes = [p.cwe for p in ok]
    maj = max(set(cwes), key=cwes.count) if cwes else ""
    return {"conformance": round(len(ok)/len(answers), 3),
            "majority_class": maj,
            "accuracy_by_always_guessing_majority":
                round(cwes.count(maj)/len(cwes), 3) if cwes else 0}
print("\nwithout any capability at all:", gameable(STRUCTURED))
assert rep2.conformance == 1.0 and rep2.expert_accuracy < 0.7

## 11 · And one run tells you almost nothing

Almost every published harness result is a single run of a stochastic system.

In [ ]:
import random

def run_once(reliability, rng):
    return rng.random() < reliability

def measure(reliability, k=5, trials=2000, seed=3):
    rng = random.Random(seed)                  # seeded: identical every run
    at_least_one = every_time = 0
    for _ in range(trials):
        outcomes = [run_once(reliability, rng) for _ in range(k)]
        at_least_one += any(outcomes)
        every_time   += all(outcomes)
    return at_least_one / trials, every_time / trials

print(f"{'per-run':>9}{'pass@5':>10}{'pass^5':>10}  what it means unattended")
for r in (0.95, 0.90, 0.80, 0.60, 0.50):
    at_k, pow_k = measure(r)
    note = ("dependable" if pow_k > .8 else
            "coin flip" if pow_k > .3 else "fails most nights")
    print(f"{r:>8.0%}{at_k:>10.1%}{pow_k:>10.1%}  {note}")
print()
print("At 80% per-run the same harness is 99.9% reliable if a human picks the")
print("good answer, and 33% reliable if nobody is looking.")

## 12 · Where it breaks — the demo that was a lucky run

In [ ]:
def one_demo(reliability, seed):
    return run_once(reliability, random.Random(seed))

demos = [one_demo(0.6, s) for s in range(12)]
print("twelve single-run demos of the same 60% harness:")
print("   " + " ".join("PASS" if d else "fail" for d in demos))
print(f"   -> {demos.count(True)} passed")
print()
print("Publish any of the passes. Every one is an honest single run. None of")
print("them is a measurement, and the reader has no way to tell which they got.")
assert demos.count(True) and demos.count(False)

## 13 · Variance, and why it precedes every other question

In [ ]:
def findings_per_run(base, noise, rng):
    return max(0, int(rng.gauss(base, noise)))

def variance_profile(base, noise, runs=30, seed=11):
    rng = random.Random(seed)
    xs = [findings_per_run(base, noise, rng) for _ in range(runs)]
    mean = sum(xs) / len(xs)
    sd = (sum((x - mean) ** 2 for x in xs) / len(xs)) ** 0.5
    return xs, mean, sd

for noise in (0.5, 3.0):
    xs, mean, sd = variance_profile(8, noise)
    print(f"noise sd={noise}:  mean {mean:.1f}  sd {sd:.2f}  "
          f"range {min(xs)}-{max(xs)}")
    print(f"   runs: {xs[:12]} ...")

_, m_stable, sd_stable = variance_profile(8, 0.5)
_, m_noisy,  sd_noisy   = variance_profile(8, 3.0)
improvement = 1.5
print()
print(f"suppose a change adds {improvement} findings on average.")
print(f"   against sd {sd_stable:.2f}: visible after a handful of runs")
print(f"   against sd {sd_noisy:.2f}: indistinguishable from a quiet Tuesday")
print()
print("Until variance is characterised, every A/B comparison you run is")
print("measuring the dice.")
assert sd_noisy > sd_stable

## 14 · The control — separate harness failure from model failure

Before changing either one, find out which is moving.

In [ ]:
def attribute(runs_same_model_same_harness, runs_same_model_new_harness):
    """If output changes when only the harness changed, it was the harness."""
    a = sum(runs_same_model_same_harness) / len(runs_same_model_same_harness)
    b = sum(runs_same_model_new_harness) / len(runs_same_model_new_harness)
    return a, b, ("harness" if abs(b - a) > 0.15 else "not the harness")

rng = random.Random(5)
baseline = [run_once(0.6, rng) for _ in range(200)]
better_harness = [run_once(0.85, rng) for _ in range(200)]   # same model, better verifier
a, b, verdict = attribute(baseline, better_harness)
print(f"same model, original harness : {a:.0%}")
print(f"same model, better verifier  : {b:.0%}")
print(f"attribution                  : {verdict}")
print()
print("Twenty-five points of reliability, no model change. Teams routinely")
print("spend that budget on a bigger backbone instead, because the harness")
print("was never measured separately.")
assert verdict == "harness"

## What you just proved

Four security domains run through one skeleton, differing only in the oracle and the blast radius, and the oracle everyone reaches for — the model's own agreement — is the one that cannot gate an action. Evaluation then separates conformance from accuracy: a harness scoring 100% on schema validity scores far lower on correctness, and matching findings on bare filename rather than parent-plus-filename silently randomises the result. Finally a harness at 80% per-run reliability shows pass@5 of 99.97% and pass^5 of 33%, and the cost per confirmed finding lands well above the cost per run.

## Your turn

Compute pass^k for one harness you run unattended, using k = the number of runs between human reviews. If you have never measured per-run reliability, that is the measurement to take first, because every other number you quote is conditioned on it.

## Where this leaves you

**What you can do now.** A harness you can name the eight parts of, a verifier you can rank against the three weaker kinds, tool signatures that cannot express the question you do not want asked, a delegation depth enforced where the orchestrator cannot lie about it, a backbone you could swap this week, and the two numbers that decide whether any of it can be left alone.

**What you still cannot do.** You have built the thing that does work, and nothing that decides what work to do. Pointed at CyberTravels' repository it would review whatever it happened to open first — which for a four-million-line estate is the same as reviewing nothing.

**Chapter 5 is the pipeline that decides: fifteen stages, in order, from git history to a severity somebody acts on. Next → B2.0, what an AI SDLC means.**

---

**Next → [B2.0 · Start here — what an AI SDLC means](https://spbreed.github.io/cyber-commons/lessons/B2.0.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B1.4.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B1.4.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*